# Agentic Rag with LangGraph

In [1]:
import os
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings


e:\Agentic AI\RAGS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../data/.env")

True

In [3]:
# Set your OpenAI API key
os.environ["OPENAI_API_KEY"] = os.getenv("API_KEY")

# Initialize Models
llm = ChatOpenAI(model="gpt-4.1", temperature=0)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3770.70it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
llm

ChatOpenAI(profile={'name': 'GPT-4.1', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x00000254B27A50D0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000254B3D16F00>, root_client=<openai.OpenAI object at 0x0000025482B2BE30>, root_async_client=<openai.AsyncOpenAI object at 0x00000254B3838350>, model_name='gpt-4.1', temperature=0.0, model_kwargs={}, openai_api_

## State Defination

In [5]:
class AgentState(TypedDict):
    name: str
    documents:List[Document]
    answer: str
    needs_retrieval: bool

In [6]:
### Sample Document and VectorStore
#Sample documents for demmonstration
sample_text = ["LangGraph: A Modern Framework for Building Intelligent AI Workflows In recent years, artificial intelligence has evolved rapidly, especially with the development of large language models (LLMs). While these models are powerful, building structured and reliable applications around them can be challenging.",
               "Retrieval-Augmented Generation (RAG): Enhancing AI with Real-Time Knowledge Retrieval-Augmented Generation, commonly known as RAG, is an advanced technique in artificial intelligence that improves the performance of language models by combining information retrieval with text generation. ",
               "Vector Databases: Powering Intelligent Search in Modern AI Systems In the era of artificial intelligence and machine learning, the need to handle complex and unstructured data has grown significantly. One of the most important innovations addressing this need is the vector database. "]

documents = [Document(page_content=text) for text in sample_text]

#Create VectorStore
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(k=3)
# retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

### Agents Function

In [7]:
def decide_retrieval(state: AgentState) -> AgentState:
    """Decide If we need to retrieve documents based on the question: """
    question = state["question"]
    
    # Simple heuristic: if the question contains certain keywords, we decide to retrieve documents  
    retrieval_keywords = ["what", "how", "explain", "describe", "tell me"]
    needs_retrieval = any(keyword in question.lower() for keyword in retrieval_keywords)
    
    return{**state, "needs_retrieval": needs_retrieval}

In [8]:
def retrieve_documents(state: AgentState) -> AgentState:
    """Retrieve relevant documents based on the question: """
    question = state["question"]
    
    # Retrieve relevant documents using the retriever
    documents = retriever.invoke(question)
    
    return {**state, "documents": documents}

In [9]:
def generate_answer(state: AgentState) -> AgentState:
    """Generate an answer based on the question and retrieved documents: """
    question = state["question"]
    documents = state.get("documents", [])
    
    # Combine the content of the retrieved documents
    if documents:
        context = "\n\n".join(doc.page_content for doc in documents)
    
    # Create a prompt for the LLM
        prompt = f"""Based on the following Context, Answer the question 
        question: {question}
        Context:{context}
        Answer:"""
    else:
        prompt = f"""Answer the following question: {question}"""
    # Generate an answer using the LLM
    response = llm.invoke(prompt)
    answer = response.content
    return {**state, "answer": answer}

### Conditional Logic

### Build the Graph

### Test the Agentic System